# Bosch Defect-Risk Classifier

LineSight — Accenture Innovation Challenge 2026, DigitalTwin.ai

Predicts which parts will fail quality control (`Response = 1`) from the Bosch Production Line Performance dataset, using only numeric features plus a time ordering derived from the date file. Categorical features are deliberately excluded — see the note at the bottom for why.

**Before running:** edit the `DATA_DIR` path in the next cell to wherever you saved the three Kaggle files (`train_numeric.csv`, `train_date.csv`; `train_categorical.csv` is not used). Everything else runs as-is.

**A note on what "PASS" means here:** unlike the plant simulator phases, there's no pre-known correct answer to check your output against — this model trains on real, messy, extremely imbalanced production data, and the actual number that comes out is the real result, whatever it is. The acceptance test is simply: does the model's PR-AUC clearly beat the no-skill baseline (the raw positive rate, ~0.0058)? A large multiple over baseline (even if the absolute PR-AUC number still looks small — that's normal for a 0.58%-positive problem) is a pass.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_recall_curve

# ---- EDIT THIS ----
DATA_DIR = r"C:\Users\ttech\Projects\linesight\data\bosch"   # folder containing the Kaggle CSVs
NUMERIC_PATH = DATA_DIR + r"\train_numeric.csv"
DATE_PATH = DATA_DIR + r"\train_date.csv"

# How many rows to load from train_numeric.csv. The full file is ~1.18M
# rows / ~2.1GB -- a few hundred thousand rows is plenty to demonstrate
# the technique (per the sprint plan) and keeps this runnable on a laptop.
# Increase if your machine handles it comfortably; decrease if it's slow
# or you run out of memory.
N_ROWS = 600_000

SEED = 0

## Load data

`train_numeric.csv` is read directly with `nrows=`, since it's the file we actually need in full width (it has every numeric feature plus the `Response` label).

`train_date.csv` is read in **chunks** and immediately reduced to just `Id` + the row's earliest timestamp (a proxy for when that part entered the line). This avoids ever holding the full ~1,150-column date file in memory at once — it's only used to establish chronological order, never as a model feature, so there's no reason to keep more than that.

In [47]:
numeric_df = pd.read_csv(NUMERIC_PATH, nrows=N_ROWS, low_memory=False)
print("numeric_df:", numeric_df.shape)
print("positive rate:", numeric_df["Response"].mean())

ParserError: Error tokenizing data. C error: out of memory

In [ ]:
def compute_start_times(date_path, ids_to_keep, chunksize=50_000):
    """Row-wise earliest timestamp per Id, read in chunks so the full
    wide date file is never held in memory at once. Filters to only the
    Ids present in numeric_df as it goes."""
    ids_to_keep = set(ids_to_keep)
    parts = []
    for chunk in pd.read_csv(date_path, chunksize=chunksize):
        chunk = chunk[chunk["Id"].isin(ids_to_keep)]
        if len(chunk) == 0:
            continue
        feature_cols = [c for c in chunk.columns if c != "Id"]
        start_time = chunk[feature_cols].min(axis=1)
        parts.append(pd.DataFrame({"Id": chunk["Id"], "start_time": start_time}))
    return pd.concat(parts, ignore_index=True)

start_times = compute_start_times(DATE_PATH, numeric_df["Id"])
print("start_times:", start_times.shape)

start_times: (800000, 2)


In [ ]:
merged = numeric_df.merge(start_times, on="Id", how="left")
n_missing_time = merged["start_time"].isna().sum()
if n_missing_time > 0:
    print(f"Dropping {n_missing_time} rows with no recoverable timestamp")
    merged = merged.dropna(subset=["start_time"])

merged = merged.sort_values("start_time").reset_index(drop=True)
feature_cols = [c for c in numeric_df.columns if c not in ("Id", "Response")]
print(f"{len(feature_cols)} numeric features, {len(merged)} rows after alignment")

Dropping 441 rows with no recoverable timestamp


MemoryError: Unable to allocate 5.77 GiB for an array with shape (969, 799559) and data type float64

## Time-based split

Split by time, not randomly — the natural ordering is what `start_time` reconstructs. This avoids the leakage a random split would risk (a random split could let a part's near-duplicate or closely related unit sit on both sides of train/test, since production runs cluster in time).

In [ ]:
cutoff = int(len(merged) * 0.7)
train_df = merged.iloc[:cutoff].copy()
test_df = merged.iloc[cutoff:].copy()

print(f"Train: {len(train_df)} rows, {train_df['Response'].sum()} defects "
      f"({train_df['Response'].mean()*100:.3f}%)")
print(f"Test:  {len(test_df)} rows, {test_df['Response'].sum()} defects "
      f"({test_df['Response'].mean()*100:.3f}%)")

Train: 419723 rows, 2807 defects (0.669%)
Test:  179882 rows, 699 defects (0.389%)


## Train

XGBoost handles missing values natively (Bosch's numeric data is heavily sparse — most parts only pass through a subset of stations), so no imputation step is needed. `scale_pos_weight` is set from the training set's actual class ratio, not assumed from the ~0.58% figure in the competition description, since the real ratio in this specific time-ordered subset may differ slightly.

In [ ]:
pos = (train_df["Response"] == 1).sum()
neg = (train_df["Response"] == 0).sum()
print(f"Train class balance: {pos} positive / {neg} negative "
      f"(scale_pos_weight = {neg/max(pos,1):.1f})")

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=neg / max(pos, 1),
    eval_metric="aucpr",
    random_state=SEED,
)
model.fit(train_df[feature_cols], train_df["Response"])
print("done")

Train class balance: 2807 positive / 416916 negative (scale_pos_weight = 148.5)
done


## Evaluate

PR-AUC, never accuracy — with a ~99.4% negative class, a model that predicts "no defect" for everything scores ~99.4% accuracy while being completely useless. PR-AUC's own no-skill baseline is the positive rate itself, which is the correct comparison point.

In [ ]:
proba = model.predict_proba(test_df[feature_cols])[:, 1]
y_test = test_df["Response"].values

model_pr_auc = average_precision_score(y_test, proba)
baseline_pr_auc = y_test.mean()

print(f"Model PR-AUC:    {model_pr_auc:.4f}")
print(f"Baseline PR-AUC: {baseline_pr_auc:.4f}  (no-skill, equals the test set's positive rate)")
print(f"Improvement over baseline: {model_pr_auc/baseline_pr_auc:.1f}x")
print()
print("PASS" if model_pr_auc > baseline_pr_auc else "FAIL",
      "- model beats the trivial baseline" if model_pr_auc > baseline_pr_auc
      else "- something is wrong, model should always beat a no-skill baseline")

Model PR-AUC:    0.0284
Baseline PR-AUC: 0.0039  (no-skill, equals the test set's positive rate)
Improvement over baseline: 7.3x

PASS - model beats the trivial baseline


In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, proba)
precision, recall = precision[:-1], recall[:-1]   # drop the trivial (recall=0, precision=1) endpoint

def recall_at_precision(target):
    ok = precision >= target
    return float(recall[ok].max()) if ok.any() else None

n_defects = int(y_test.sum())

for target in (0.20, 0.40):
    r = recall_at_precision(target)
    if r is not None:
        caught = int(round(r * n_defects))
        print(f"At {target*100:.0f}% precision: catches {r*100:.0f}% of real defects "
              f"({caught} of {n_defects}) -- roughly {round(1/target)} flagged "
              f"inspections per real defect found, versus inspecting nothing "
              f"until end-of-line.")
    else:
        print(f"{target*100:.0f}% precision is not reached on this test set.")

best_idx = int(np.argmax(precision))
print(f"\nBest precision actually achieved: {precision[best_idx]*100:.1f}% "
      f"(catching {recall[best_idx]*100:.0f}% of defects at that point)")

At 20% precision: catches 1% of real defects (8 of 699) -- roughly 5 flagged inspections per real defect found, versus inspecting nothing until end-of-line.
At 40% precision: catches 0% of real defects (3 of 699) -- roughly 2 flagged inspections per real defect found, versus inspecting nothing until end-of-line.

Best precision actually achieved: 60.0% (catching 0% of defects at that point)


In [ ]:
for target in (0.05, 0.10, 0.15, 0.20):
    r = recall_at_precision(target)
    if r is not None:
        caught = int(round(r * n_defects))
        print(f"At {target*100:.0f}% precision: catches {r*100:.1f}% of defects "
              f"({caught} of {n_defects})")
    else:
        print(f"{target*100:.0f}% precision not reached")

At 5% precision: catches 16.6% of defects (116 of 699)
At 10% precision: catches 15.7% of defects (110 of 699)
At 15% precision: catches 2.1% of defects (15 of 699)
At 20% precision: catches 1.1% of defects (8 of 699)


## Notes for the README / proposal

- **Categorical features (`train_categorical.csv`) were deliberately not used.** Numeric features alone are already hundreds of columns and sufficient to demonstrate the technique within this sprint's time budget; Bosch's categorical data is notoriously high-cardinality and sparse, and would add real complexity for modest expected gain. Documented here as a scope decision, not an oversight — a natural next step, not attempted this round.
- **The "magic feature" several public leaderboard solutions used (differencing `start_time` and Id between consecutive sorted rows) was deliberately not used either.** It exploits an artifact of how Kaggle constructed the train/test split for this specific competition, and multiple practitioners writing about this dataset explicitly flag it as bordering on data leakage in a real deployment sense — not something to build a real business case on, even though it would inflate the score.
- **This is only a subset of the real 1.18M-row dataset** (`N_ROWS` above). The full dataset would very likely produce a stronger result; report what actually ran, and say plainly that this is a subset if a judge asks.
- **Sanity-check context, not a target to hit:** the competition's actual metric was Matthews Correlation Coefficient, not PR-AUC — the top 10% of the original Kaggle leaderboard scored ~0.44 MCC, and a solid XGBoost solution without special feature engineering scored ~0.36. One published paper reports individual-model ROC-AUC around 0.72. These aren't directly comparable to PR-AUC, but they indicate a well-built model on this data should land clearly above baseline, not marginally.